<h1><img src='assets/nephroq_logo.png' alt='NephroQ logo' width='40'> NephroQ — interactive renal-risk digital twin · gemelo digital interactivo</h1>

**EN** — Enter a patient's markers (blood + urine + blood pressure) or load an example, and see the projected eGFR trajectory, the modeled time to the eGFR < 15 threshold, and the KDIGO GFR category. Same engine and example patients as the web app.

**ES** — Ingresa los marcadores del paciente (sangre + orina + presión arterial) o carga un ejemplo, y observa la trayectoria proyectada de la eGFR, el tiempo modelado hasta el umbral eGFR < 15 y la categoría de TFG de KDIGO. Utiliza el mismo motor y los mismos pacientes de ejemplo que la aplicación web.

> ⚠️ **Research prototype (TRL 4) — NOT a diagnostic tool / Prototipo de investigación (TRL 4) — NO es una herramienta diagnóstica.**
>
> ▶️ Run the two cells below (**Runtime → Run all** / *Entorno de ejecución → Ejecutar todo*). The code is hidden — just use the controls that appear.


In [ ]:
#@title ⚙️ Setup — run first / ejecutar primero { display-mode: "form" }
import os, sys, subprocess

REPO = "https://github.com/Danpc11/nephroq.git"

def _ensure_src():
    for cand in ("src", "nephroq/src", "repo/src"):
        if os.path.isdir(cand):
            sys.path.insert(0, os.path.abspath(cand))
    try:
        import i18n  # noqa: F401
        return
    except Exception:
        pass
    if not os.path.isdir("nephroq"):
        subprocess.run(["git", "clone", "-q", REPO], check=False)
    sys.path.insert(0, os.path.abspath("nephroq/src"))

_ensure_src()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "scipy", "matplotlib", "ipywidgets"], check=False)
try:
    from google.colab import output as _co
    _co.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from egfr_measurement import egfr_cr, egfr_cr_cys
from mechanistic_twin import MechanisticRenalModel, N_of_egfr, DIALYSIS_eGFR, gfr_category
from i18n import t as T, PRESETS, LANGUAGES, preset_by_id

# Public research calibration (hierarchical Bayesian fit; see docs/MODEL_DOCUMENTATION.md).
Q_POP, KHF_POP = 1.52, 0.0141
W_POP = np.array([0.0144, 0.0180, 0.0108])
print("Setup complete — run the next cell / listo, ejecuta la siguiente celda.")


In [ ]:
#@title 🩺 NephroQ — interactive dashboard / panel interactivo { display-mode: "form" }
import re

def _md(s):
    return re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", s)

def _card(label, value, unit):
    return (f'<div style="flex:1;min-width:150px;background:#f6f7fb;border:1px solid #e6e8f0;'
            f'border-radius:12px;padding:10px 14px">'
            f'<div style="font-size:12px;color:#6b7280">{label}</div>'
            f'<div style="font-size:22px;font-weight:700;color:#111">{value} '
            f'<span style="font-size:12px;font-weight:400;color:#6b7280">{unit}</span></div></div>')

def _box(text, bg, border, icon=""):
    return (f'<div style="background:{bg};border-left:4px solid {border};border-radius:8px;'
            f'padding:10px 12px;margin:8px 0;font-family:system-ui;font-size:13px;color:#111">'
            f'{icon}{_md(text)}</div>')

state = dict(age=58, sex="F", creatinine=1.3, use_cys=False, cystatin=1.3,
             hba1c=8.1, uacr=145.0, sbp=142, treated=False, active=None)

def _project(a1c, sbp, uacr, egfr0, treated, years=15):
    m = MechanisticRenalModel(a1c=a1c, sbp=sbp, uacr=uacr, u=1.0 if treated else 0.0,
                              k_hf=KHF_POP, q=Q_POP, w_a1c=W_POP[0], w_uacr=W_POP[1], w_sbp=W_POP[2])
    t, N, egfr, td = m.simulate(N_of_egfr(egfr0), years=years)
    return t, egfr, td

results = widgets.Output()

def refresh(lang):
    s = state
    with results:
        clear_output(wait=True)
        if s["use_cys"]:
            egfr0 = egfr_cr_cys(s["creatinine"], s["cystatin"], s["age"], s["sex"])
            method = T(lang, "method_cr_cys")
        else:
            egfr0 = egfr_cr(s["creatinine"], s["age"], s["sex"])
            method = T(lang, "method_cr")
        t_a, e_a, td_a = _project(s["hba1c"], s["sbp"], s["uacr"], egfr0, s["treated"])
        t_b, e_b, td_b = _project(s["hba1c"], s["sbp"], s["uacr"], egfr0, not s["treated"])
        label_a = T(lang, "label_current_tx") if s["treated"] else T(lang, "label_no_tx")
        label_b = T(lang, "label_reno_added") if not s["treated"] else T(lang, "label_tx_stopped")
        horizon = int(t_a[-1])
        state_lbl = T(lang, "state_current") if s["treated"] else T(lang, "state_untreated")
        time_str = T(lang, "years", v=td_a) if np.isfinite(td_a) else T(lang, "gt_years", v=horizon)

        cards = ('<div style="display:flex;gap:12px;flex-wrap:wrap;font-family:system-ui;margin:2px 0 6px">'
                 + _card(T(lang, "baseline_egfr"), f"{egfr0:.1f}", "mL/min/1.73m²")
                 + _card(T(lang, "kdigo"), gfr_category(egfr0), "")
                 + _card(T(lang, "time_title", state=state_lbl), time_str, "")
                 + '</div>')
        display(HTML(cards))

        if s["active"]:
            ap = preset_by_id(s["active"])
            if ap:
                display(HTML(_box(ap["note"][lang], "#eef4ff", "#3b82f6")))

        fig, ax = plt.subplots(figsize=(8, 4.2))
        ax.plot(t_a, e_a, lw=2.5, color="#E24B4A", label=label_a)
        ax.plot(t_b, e_b, lw=2.5, color="#1D9E75", label=label_b)
        ax.axhline(DIALYSIS_eGFR, color="k", lw=1, ls="--")
        ax.text(0.3, DIALYSIS_eGFR + 2, T(lang, "plot_threshold"), fontsize=9)
        ax.set_xlabel(T(lang, "plot_x")); ax.set_ylabel(T(lang, "plot_y"))
        ax.set_title(T(lang, "plot_title")); ax.legend()
        ax.grid(alpha=0.15); fig.tight_layout(); plt.show()

        if np.isfinite(td_a) and np.isfinite(td_b):
            display(HTML(_box(T(lang, "diff_info", label=label_b, d=abs(td_b - td_a)), "#eef4ff", "#3b82f6")))
        if not s["use_cys"]:
            display(HTML(_box(T(lang, "cystatin_warning"), "#fff7ed", "#f59e0b", "⚠️ ")))
        display(HTML(_box(T(lang, "disclaimer"), "#f3f4f6", "#9ca3af")))

root = widgets.VBox()

def rebuild(*_):
    lang = LANGUAGES[lang_tb.value]

    def labeled(key, w):
        return widgets.VBox([widgets.HTML(f"<b style='font-size:13px'>{T(lang, key)}</b>"), w],
                            layout=widgets.Layout(margin="2px 0"))

    age   = widgets.BoundedIntText(value=state["age"], min=18, max=100)
    sex   = widgets.ToggleButtons(value=state["sex"], options=["F", "M"])
    creat = widgets.BoundedFloatText(value=state["creatinine"], min=0.3, max=10.0, step=0.1)
    ucys  = widgets.Checkbox(value=state["use_cys"], description=T(lang, "have_cystatin"),
                             style={"description_width": "initial"})
    cys   = widgets.BoundedFloatText(value=state["cystatin"], min=0.3, max=8.0, step=0.1,
                                     disabled=not state["use_cys"])
    a1c   = widgets.BoundedFloatText(value=state["hba1c"], min=4.0, max=15.0, step=0.1)
    uacr  = widgets.BoundedFloatText(value=state["uacr"], min=0.0, max=3000.0, step=5.0)
    sbp   = widgets.BoundedIntText(value=state["sbp"], min=80, max=220)
    trt   = widgets.Checkbox(value=state["treated"], description=T(lang, "treated"),
                             style={"description_width": "initial"})

    def bind(w, k, extra=None):
        def obs(ch):
            state[k] = ch["new"]
            if extra: extra(ch["new"])
            refresh(lang)
        w.observe(obs, "value")

    bind(age, "age"); bind(sex, "sex"); bind(creat, "creatinine")
    bind(ucys, "use_cys", lambda v: setattr(cys, "disabled", not v))
    bind(cys, "cystatin"); bind(a1c, "hba1c"); bind(uacr, "uacr"); bind(sbp, "sbp"); bind(trt, "treated")

    ex_btns = []
    for p in PRESETS:
        b = widgets.Button(description=p["label"][lang],
                           layout=widgets.Layout(width="auto"), button_style="")
        def mk(pp):
            def onclick(_):
                state.update(pp["markers"]); state["active"] = pp["id"]; rebuild()
            return onclick
        b.on_click(mk(p)); ex_btns.append(b)

    reset = widgets.Button(description=T(lang, "reset"), layout=widgets.Layout(width="auto"))
    def onreset(_):
        state.update(dict(age=58, sex="F", creatinine=1.3, use_cys=False, cystatin=1.3,
                          hba1c=8.1, uacr=145.0, sbp=142, treated=False, active=None))
        rebuild()
    reset.on_click(onreset)

    controls = widgets.VBox([
        widgets.HTML(f"<h4 style='margin:2px 0'>{T(lang, 'examples_header')}</h4>"),
        widgets.HTML(f"<div style='color:#6b7280;font-size:12px'>{T(lang, 'examples_caption')}</div>"),
        widgets.VBox(ex_btns), reset,
        widgets.HTML("<hr style='margin:8px 0'>"),
        widgets.HTML(f"<h4 style='margin:2px 0'>{T(lang, 'markers_header')}</h4>"),
        labeled("age", age), labeled("sex", sex),
        widgets.HTML(f"<b>{T(lang, 'blood')}</b>"),
        labeled("creatinine", creat), ucys, labeled("cystatin", cys), labeled("hba1c", a1c),
        widgets.HTML(f"<b>{T(lang, 'urine')}</b>"), labeled("uacr", uacr),
        widgets.HTML(f"<b>{T(lang, 'in_clinic')}</b>"), labeled("sbp", sbp),
        widgets.HTML("<hr style='margin:8px 0'>"), trt,
    ], layout=widgets.Layout(width="330px", padding="4px 10px 4px 0"))

    root.children = [widgets.HBox([controls, results],
                                  layout=widgets.Layout(align_items="flex-start"))]
    refresh(lang)

lang_tb = widgets.ToggleButtons(options=list(LANGUAGES), value="English",
                                description="", layout=widgets.Layout(margin="0 0 6px 0"))
lang_tb.observe(lambda ch: rebuild(), "value")
rebuild()
display(widgets.VBox([lang_tb, root]))


---
### 🔬 Recalibrate with your own cohort · recalibrar con tu cohorte

**EN** — With a longitudinal CSV (`patient_id, time_years, egfr, hba1c, uacr, sbp`) you can recalibrate and validate:
```
!cd nephroq/src && CKD_CSV=/content/your_data.csv python mvp_calibration.py
```
**ES** — Con un CSV longitudinal (mismas columnas) puedes recalibrar y validar con el comando de arriba.

Full docs · documentación: [github.com/Danpc11/nephroq](https://github.com/Danpc11/nephroq) — see `docs/CLINICIAN_DEMO.md` for the clinician walkthrough.

> **Notice / Aviso:** research tool (TRL4), not clinically validated. Must not be used for medical decisions without qualified supervision. / Herramienta de investigación, no validada clínicamente. No debe usarse para decisiones médicas sin supervisión calificada.
